In [ ]:
BRANCH = "feat/project-scaffold"
DRIVE = "/content/drive/MyDrive/storozha"
RUNS = f"{DRIVE}/runs"
TRAIN = False
SAHI = False
FORCE = False
ONLY = []


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
import torch
torch.cuda.is_available()
import sys
sys.path.insert(0, "/content/storozha-anti-ambush/src")
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(RUNS, exist_ok=True)

os.chdir("/content")
if not os.path.isdir("storozha-anti-ambush"):
    !git clone -q -b {BRANCH} https://github.com/Luzefik/storozha-anti-ambush.git
os.chdir("/content/storozha-anti-ambush")


!git pull -q --ff-only && git log --oneline -1
!pip install -q ultralytics huggingface_hub
!pip install -q -e . --no-deps
import sys
sys.path.insert(0, os.path.abspath("src"))
sys.path.insert(0, "/content/storozha-anti-ambush/src")

In [ ]:
from getpass import getpass
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")
!python scripts/download_data.py
!python scripts/prepare_data.py

In [ ]:
import json
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from ultralytics import RTDETR, YOLO

from zfpv.config import load_config
from zfpv.data.tiling import predict_tiled
from zfpv.eval.metrics import evaluate

cfg = load_config()

EV = cfg["eval"]
YOLO_DIR = Path(cfg["paths"]["yolo_dir"])
manifest = pd.read_csv(cfg["paths"]["split_manifest"])
DAY_NIGHT = {r.video: "night" if r.is_night else "day" for r in manifest.itertuples()}
VAL = manifest.loc[manifest.split == "val", "video"].tolist()
TEST = manifest.loc[manifest.split == "test", "video"].tolist()


def load_gt(video, every=1):

    paths = sorted((YOLO_DIR / "images" / video).glob("frame_*.png"))[::every]
    h, w = cv2.imread(str(paths[0])).shape[:2]
    boxes = []
    for p in paths:
        text = (YOLO_DIR / "labels" / video / f"{p.stem}.txt").read_text()
        rows = np.array(text.split(), dtype=float).reshape(-1, 5)
        xc, yc, bw, bh = rows[:, 1] * w, rows[:, 2] * h, rows[:, 3] * w, rows[:, 4] * h
        boxes.append(np.stack([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], axis=1))
    return paths, boxes


def run_eval(weights, videos, imgsz=640, tile=None, every=1, batch=16):

    model = (RTDETR if "rtdetr" in str(weights) else YOLO)(weights)
    seconds, n_frames = 0.0, 0

    def predict(images):
        res = model.predict(images, imgsz=imgsz, conf=0.001, max_det=EV["max_detections"],
                            device="mps", verbose=False)
        return [np.hstack([r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()[:, None]])
                for r in res]

    preds, gts = {}, {}
    for v in videos:
        paths, gts[v] = load_gt(v, every)
        preds[v] = []
        for i in range(0, len(paths), batch):
            frames = [cv2.imread(str(p)) for p in paths[i:i + batch]]
            t0 = time.perf_counter()
            preds[v] += [predict_tiled(predict, f, tile) for f in frames] if tile else predict(frames)
            seconds, n_frames = seconds + time.perf_counter() - t0, n_frames + len(frames)
    res = evaluate(preds, gts, EV["iou_thresholds"], EV["conf_threshold"],
                   EV["max_detections"], groups=DAY_NIGHT)
    res["ms_per_frame"] = 1000 * seconds / max(n_frames, 1)
    return res


def log_run(name, runs=RUNS):

    df = pd.read_csv(f"{runs}/{name}/results.csv")
    df.columns = df.columns.str.strip()
    best = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
    row = {"run": name, "epochs_run": len(df), "best_epoch": int(best["epoch"]),
           "best_mAP50": round(best["metrics/mAP50(B)"], 4),
           "last10_mAP50": round(df["metrics/mAP50(B)"].tail(10).mean(), 4)}
    log = Path(runs) / "summary.csv"
    pd.DataFrame([row]).to_csv(log, mode="a", header=not log.exists(), index=False)


print(f"val: {VAL}\ntest: {TEST} (не чіпаємо)")

In [ ]:
EXPERIMENTS = [

    dict(name="exp005-yolo26n-1280-scale09", model="yolo26n.pt", imgsz=1280, batch=-1,
         cache="disk", scale=0.9),
    dict(name="exp007-yolov8s-p2-1280", model="yolov8s-p2.yaml", pretrained="yolov8s.pt",
         imgsz=1280, batch=-1, cache="disk"),
    dict(name="exp008-yolo26s-1280", model="yolo26s.pt", imgsz=1280, batch=-1, cache="disk"),
    dict(name="exp009-yolo26n-1280-freeze10", model="yolo26n.pt", imgsz=1280, batch=-1,
         cache="disk", freeze=10),
    dict(name="exp010-yolo11n-1280", model="yolo11n.pt", imgsz=1280, batch=-1, cache="disk"),

    dict(name="exp011-yolo26s-1280-freeze10", model="yolo26s.pt", imgsz=1280, batch=-1,
         cache="disk", freeze=10),
    dict(name="exp012-yolo26n-1280-freeze5", model="yolo26n.pt", imgsz=1280, batch=-1,
         cache="disk", freeze=5),
    dict(name="exp013-yolo26n-1280-freeze20", model="yolo26n.pt", imgsz=1280, batch=-1,
         cache="disk", freeze=20),
]

CV_EXPERIMENTS = [
    dict(name="cv-yolo26n-1280", model="yolo26n.pt", imgsz=1280, batch=-1, cache="disk"),
    dict(name="cv-yolo26n-1280-freeze10", model="yolo26n.pt", imgsz=1280, batch=-1,
         cache="disk", freeze=10),
]
K = 4


def unfinished(last):

    return torch.load(last, map_location="cpu", weights_only=False).get("optimizer") is not None


def train(exp, data=None, project=RUNS):
    exp = dict(exp)
    name, weights = exp.pop("name"), exp.pop("model")
    last = Path(project) / name / "weights" / "last.pt"
    Model = RTDETR if "rtdetr" in weights else YOLO
    if last.exists() and not unfinished(last):

        return
    if last.exists():

        Model(str(last)).train(resume=True)
    else:
        Model(weights).train(data=str(data or YOLO_DIR / "data.yaml"), epochs=100, patience=20,
                             seed=42, device="mps", workers=2, project=str(project), name=name,
                             exist_ok=True, **exp)
    log_run(name, runs=str(project))


def cv_folds(k=K):

    pool = manifest[manifest.split != "test"]
    order = pd.concat([pool[pool.is_night],
                       pool[~pool.is_night].sort_values("n_frames", ascending=False)])
    folds = [[] for _ in range(k)]
    for i, v in enumerate(order.video):
        folds[i % k].append(v)
    return folds


def write_fold_yamls(folds):

    def lines(videos):
        return [f"./images/{v}/{p.name}" for v in videos
                for p in sorted((YOLO_DIR / "images" / v).glob("frame_*.png"))]

    paths = []
    for k, val_videos in enumerate(folds):
        train_videos = [v for f in folds for v in f if v not in val_videos]
        (YOLO_DIR / f"fold{k}_train.txt").write_text("\n".join(lines(train_videos)) + "\n")
        (YOLO_DIR / f"fold{k}_val.txt").write_text("\n".join(lines(val_videos)) + "\n")
        (YOLO_DIR / f"fold{k}.yaml").write_text(yaml.safe_dump(
            {"train": f"fold{k}_train.txt", "val": f"fold{k}_val.txt",
             "names": {0: "waiter_fpv_drone"}}, sort_keys=False))
        paths.append(YOLO_DIR / f"fold{k}.yaml")
    return paths


if TRAIN:
    for exp in EXPERIMENTS:
        train(exp)
    fold_yamls = write_fold_yamls(cv_folds())
    for exp in CV_EXPERIMENTS:
        for k, data in enumerate(fold_yamls):
            train({**exp, "name": f"fold{k}"}, data=data, project=Path(RUNS) / exp["name"])


In [ ]:
def discover(runs_dir):
    rows = []
    for best in sorted(Path(runs_dir).glob("*/weights/best.pt")):
        run = best.parent.parent
        args = yaml.safe_load((run / "args.yaml").read_text())
        hist = pd.read_csv(run / "results.csv")
        hist.columns = hist.columns.str.strip()
        rows.append(dict(run=run.name, weights=str(best), model=Path(str(args["model"])).stem,
                         imgsz=args["imgsz"], data=str(args["data"]), epochs=len(hist),
                         tiles="tiles" in str(args["data"]),
                         sig=tuple(hist["metrics/mAP50(B)"].round(4).head(5))))
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df = df[~df.data.str.contains("fold")]          # фолди CV оцінюємо окремо, у розділі 5
    df = df.sort_values("epochs", ascending=False).drop_duplicates(["model", "imgsz", "sig"])
    return df.drop(columns="sig").sort_values("run").reset_index(drop=True)


runs = discover(RUNS)
if ONLY:
    runs = runs[runs.run.isin(ONLY)]
runs

In [ ]:
def evaluate_run(r, tile=None):
    out = Path(r.weights).parent.parent / f"eval_val{f'_sahi{tile}' if tile else ''}.json"
    if out.exists() and not FORCE:
        return json.loads(out.read_text())
    print(f"оцінюю {r.run}" + (f" (SAHI {tile})" if tile else ""))
    res = run_eval(r.weights, VAL, imgsz=640 if tile else int(r.imgsz), tile=tile)
    out.write_text(json.dumps(res, indent=2))
    return res


results = {}
for r in runs.itertuples():
    if r.tiles:
        results[r.run, "SAHI 640"] = evaluate_run(r, tile=640)
        continue
    results[r.run, "цілий кадр"] = evaluate_run(r)
    if SAHI:
        results[r.run, "SAHI 640"] = evaluate_run(r, tile=640)
print(len(results), "оцінок")

In [ ]:
def summary_row(res):
    o, g = res["overall"], res["per_group"]
    per_video = [v["AP@0.5"] for v in res["per_video"].values() if v["n_gt"]]
    return {
        "AP@0.5": o["AP@0.5"], "AP@0.75": o["AP@0.75"],
        "precision": o["precision@0.5"], "recall": o["recall@0.5"],
        "recall день": g.get("day", {}).get("recall@0.5"),
        "recall ніч": g.get("night", {}).get("recall@0.5"),
        "AP@0.5 по відео": f"{np.nanmean(per_video):.2f} ± {np.nanstd(per_video):.2f}",
        "мс/кадр": res.get("ms_per_frame"),
    }


table = pd.DataFrame({k: summary_row(v) for k, v in results.items()}).T
table.index.names = ["запуск", "режим"]
table = table.sort_values("AP@0.5", ascending=False)
table.round(3)

In [ ]:
per_video = pd.DataFrame({k: {v: m["AP@0.5"] for v, m in res["per_video"].items()}
                          for k, res in results.items()}).T
per_video.index.names = ["запуск", "режим"]
per_video.columns = [f"{v} ({DAY_NIGHT.get(v, '?')})" for v in per_video.columns]
per_video.loc[table.index].style.format("{:.2f}").background_gradient(cmap="Blues", vmin=0, vmax=1)

In [ ]:
labels = [f"{run} · {mode}" for run, mode in table.index]
fig, ax = plt.subplots(figsize=(8, 0.45 * len(table) + 1))
ax.barh(labels[::-1], table["AP@0.5"].astype(float)[::-1], color="#2a78d6")
for y, v in enumerate(table["AP@0.5"].astype(float)[::-1]):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel("AP@0.5 на val (спільний evaluator)")
ax.set_title("Порівняння моделей", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
FOLDS = cv_folds()
cv_rows = []
for cv_dir in sorted(p for p in Path(RUNS).glob("cv-*") if p.is_dir()):
    for fold_dir in sorted(cv_dir.glob("fold*")):
        best = fold_dir / "weights" / "best.pt"
        if not best.exists():
            continue
        k = int(fold_dir.name.removeprefix("fold"))
        out = fold_dir / "eval_fold_val.json"
        if out.exists() and not FORCE:
            res = json.loads(out.read_text())
        else:
            imgsz = yaml.safe_load((fold_dir / "args.yaml").read_text())["imgsz"]
            res = run_eval(str(best), FOLDS[k], imgsz=imgsz)
            out.write_text(json.dumps(res, indent=2))
        cv_rows.append({"cv": cv_dir.name, "fold": k, "відео": ", ".join(FOLDS[k]),
                        "AP@0.5": res["overall"]["AP@0.5"], "recall": res["overall"]["recall@0.5"]})

cv_table = pd.DataFrame(cv_rows)
if len(cv_table):
    display(cv_table.round(3))
    display(cv_table.groupby("cv")[["AP@0.5", "recall"]].agg(["mean", "std"]).round(3))
else:
    print("Папок cv-* з моделями немає")

In [ ]:
stamp = time.strftime("%Y-%m-%d")
table.to_csv(f"{RUNS}/eval_summary_{stamp}.csv")
per_video.to_csv(f"{RUNS}/eval_per_video_{stamp}.csv")
if len(cv_table):
    cv_table.to_csv(f"{RUNS}/eval_cv_{stamp}.csv", index=False)
